#  Parallax

**Parallax** is the apparent change in the position of an object against a more distant
background when the observer (or camera) **moves**. The key word is *moves*: parallax
is caused by a change in the **viewpoint's position** (a translation), not by simply
looking in a different direction.

Nearby objects shift a lot; distant objects shift only a little. This is why, when
you look out of a moving car, the fence next to the road races past while the far
mountains barely seem to move. That difference in apparent speed *is* parallax, and
it is the everyday cue our brain (and a computer-vision system) uses to sense depth.

### Parallax is inversely proportional to depth

If the camera translates by a baseline $B$, a point at depth $Z$ shifts in the image by
an amount (its parallax / disparity) that is *inversely proportional* to its depth:

$$
\text{parallax} \;\propto\; \frac{B}{Z}
$$

so, equivalently, depth is recovered from parallax as

$$
Z \;\propto\; \frac{B}{\text{parallax}}.
$$

- **Near points** ($Z$ small) $\Rightarrow$ **large** parallax.
- **Far points** ($Z$ large) $\Rightarrow$ **small** parallax.
- A point infinitely far away has **zero** parallax (it does not move between views).

This inverse relationship is exactly why we can turn image motion into a depth
measurement — and why depth is hardest to estimate for far-away points, where the
parallax signal is tiny.

### Why pure rotation gives no parallax (and no depth)

If the camera only **rotates** about its optical centre and does not translate, then
*every* point — near or far — moves in the image by the same amount, determined solely
by the rotation angle. There is **no relative shift** between near and far objects, so
there is **no parallax and therefore no depth information**.

This is the reason **panorama stitching assumes a rotation-only camera model**: with no
parallax, the two views are related by a single [homography](homography.ipynb) (a pure
2D warp) and the images can be blended seamlessly. As soon as the camera translates,
parallax appears, the pure-homography assumption breaks, and near objects create
"ghosting"/misalignment in the stitched panorama.

---

In computer vision, "parallax" usually refers to the relative displacement of
corresponding points between two views of the same scene — as in stereo vision or
structure-from-motion. Because parallax encodes depth, it is a central concept in 3D
vision. Here is how it is handled in OpenCV.

---

### **1. Parallax in Stereo Vision**
In stereo vision the parallax is called the **disparity**: the difference in the pixel
position of a corresponding point between the left and right images. The disparity map
is converted directly into a depth (or distance) map.

#### **Steps to Compute Parallax (Disparity Map) in OpenCV**
1. **Calibrate the cameras**:
   Use `cv2.calibrateCamera()` to obtain the camera matrices and distortion coefficients.

2. **Rectify the images**:
   Use `cv2.stereoRectify()` to align the stereo pair so that the epipolar lines are
   horizontal (corresponding points then lie on the same image row, and disparity
   becomes a purely horizontal shift).

3. **Compute the disparity map**:
   Use a stereo matching algorithm such as:
   - `cv2.StereoBM_create()`: Block Matching.
   - `cv2.StereoSGBM_create()`: Semi-Global Block Matching.

```python
import cv2
import numpy as np

# Load a rectified stereo pair
left_img = cv2.imread("left_image.png", cv2.IMREAD_GRAYSCALE)
right_img = cv2.imread("right_image.png", cv2.IMREAD_GRAYSCALE)

# Create a StereoBM object and compute the disparity
stereo = cv2.StereoBM_create(numDisparities=16, blockSize=15)
disparity = stereo.compute(left_img, right_img)

# Normalize the disparity map for visualization
disp_vis = cv2.normalize(disparity, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
disp_vis = np.uint8(disp_vis)

cv2.imshow("Disparity Map", disp_vis)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

4. **Calculate depth**:
   Depth is computed from the disparity with

$$
Z = \frac{f \cdot B}{d}
$$

   where:
   - $f$ = focal length of the camera (in pixels).
   - $B$ = baseline (distance between the two camera centres).
   - $d$ = disparity (the parallax, in pixels).

   Note this is the same inverse relationship as above ($Z \propto B / d$): small
   disparity means a far point, large disparity means a near point.

---

### **2. Parallax in Structure-from-Motion**
In structure-from-motion (SfM), a single moving camera provides the parallax: the scene
is reconstructed in 3D from multiple 2D images taken from different positions. Enough
translation between views (a good "parallax angle") is required — views taken from
nearly the same spot, or from a purely rotating camera, cannot be triangulated reliably.

#### **Key Steps**:
1. **Detect and match features**:
   Use feature detection and matching (e.g. SIFT, ORB) to find corresponding points
   across images.

2. **Estimate the essential/fundamental matrix**:
   Use `cv2.findEssentialMat()` or `cv2.findFundamentalMat()` to relate points between
   two views.

3. **Recover the pose**:
   Use `cv2.recoverPose()` to compute the camera's relative rotation and translation.

4. **Triangulate points**:
   Use `cv2.triangulatePoints()` to compute 3D points from the matched 2D points.

---

### **Applications of Parallax**
- **Depth estimation**: generating disparity maps and depth maps.
- **3D reconstruction**: stereo and multiple-view geometry.
- **Augmented reality**: estimating scene geometry to overlay virtual objects.
- **Motion analysis**: using parallax motion to understand scene structure.

---

Refs: [Parallax images (Medium)](https://medium.com/analytics-vidhya/parallax-images-14e92ebb1bae)
